# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jh-emon002/flyrank-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane**: Refresh / Content Opportunity Scoring

**Primary ML task**: Ranking / scoring

The practical goal is to rank content items by how strongly they deserve human review, rather than only classify each page as good or bad. An SEO/content reviewer has limited time and needs to know which pages should be inspected first. Therefore, the final output should be an ordered review queue in which higher-ranked pages represent stronger review opportunities.

A classification model may later be used internally to estimate the probability of an observed outcome such as decline, but the decision-facing output is still a ranking because the reviewer acts on the top candidates first.

In [2]:
from pathlib import Path
import pandas as pd
import subprocess

repo = Path("/content/flyrank-intern")

if not (repo / "data/raw/content_refresh_anonymized.csv").exists():
    subprocess.run(
        [
            "git", "clone", "-q",
            "https://github.com/jh-emon002/flyrank-intern.git",
            str(repo)
        ],
        check=True
    )

data_path = repo / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

Dataset shape: (30000, 44)
Unique content items: 30000
Unique clients: 32


## 2. Target or proxy

For this Week-2 framing exercise, I will use the starter dataset's `trend_direction == "down"` field to construct a temporary binary proxy called `is_declining_proxy`. A value of 1 means that the content item's search impressions declined by the starter definition, while 0 means it did not.

This is not an ideal final target. It is derived from the current snapshot and therefore represents a rule-defined proxy rather than a genuinely future observed outcome. I will treat it only as a convenient Week-2 target sketch.

A stronger eventual target would use a separated time window, for example:

prior 90-day observable features → decline during the following 30 days

That would allow the system to estimate future review risk without using information from the outcome period itself.

In [3]:
lane_df = df.copy()

lane_df["is_declining_proxy"] = (
    lane_df["trend_direction"].eq("down").astype(int)
)

target_counts = lane_df["is_declining_proxy"].value_counts().sort_index()

print(target_counts)

print(
    f"\nProxy-positive rate: "
    f"{lane_df['is_declining_proxy'].mean():.1%}"
)


is_declining_proxy
0    13738
1    16262
Name: count, dtype: int64

Proxy-positive rate: 54.2%


In [4]:
LEAKAGE_COLUMNS = ["trend_direction", "trend_pct"]

print(
    "Do not use as model features when predicting "
    "is_declining_proxy:",
    LEAKAGE_COLUMNS
)

Do not use as model features when predicting is_declining_proxy: ['trend_direction', 'trend_pct']


## 3. Success metric

My primary success metric will be **Precision@20** because the output is a ranked human-review queue. Precision@20 measures the proportion of the 20 highest-ranked content items that meet the chosen target.

I will use a transparent fixed-rule baseline as the comparison point. A model will only count as useful if its Precision@20 is meaningfully higher than the baseline on the same evaluation data. As a provisional working criterion, I would look for an improvement of at least 10 percentage points in Precision@20 over the fixed-rule baseline.

The exact threshold is a decision-policy choice rather than a universal definition of good performance, so it may later be adjusted based on real reviewer capacity and the quality of the final future-outcome target.

In [5]:
K = 20

base_rate = lane_df["is_declining_proxy"].mean()

print(f"Review capacity K: {K}")
print(f"Current proxy base rate: {base_rate:.1%}")
print(
    "Primary evaluation metric: Precision@20\n"
    "Working success criterion: beat the fixed-rule "
    "baseline Precision@20 by >= 0.10"
)


Review capacity K: 20
Current proxy base rate: 54.2%
Primary evaluation metric: Precision@20
Working success criterion: beat the fixed-rule baseline Precision@20 by >= 0.10


## 4. The unit of analysis, as a real dataframe

**Unit of analysis**: one content item/page.

Each dataframe row represents one pseudonymized content item belonging to one pseudonymized client. The observable columns describe characteristics and performance of that content item over the starter dataset's trailing 90-day measurement window.

The ranking system would therefore assign one review-priority score to each eligible content item.

In [6]:
display_cols = [
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "is_declining_proxy",
]

lane_df[display_cols].head(10)


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,0.76,10.6,187,20,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,0.05,20.3,445,25,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,0.09,36.5,141,20,3515.0,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,0.49,6.2,463,22,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,0.13,44.0,263,14,2803.0,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,5,0.03,8.5,147,20,3080.0,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,1,0.00,7.0,90,20,3059.0,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,28,0.06,21.2,445,22,NaN,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,32574,29,68,0.09,46.0,90,20,3807.0,1
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,1240,2,3,0.16,4.9,257,104,NaN,1


In [7]:
print("Rows:", len(lane_df))
print("Unique content IDs:", lane_df["content_id"].nunique())
print(
    "Duplicate content IDs:",
    lane_df["content_id"].duplicated().sum()
)

Rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0


## 5. Why ML beats a fixed rule here

A fixed rule is an important baseline, but it may not be sufficient for the final prioritization decision. Individual rules can identify obvious cases, such as pages that are declining while still receiving meaningful impressions, but they can leave thousands of candidates with the same recommendation.

In my Week-1 exploration, 13,152 content items were both currently declining and receiving at least 100 impressions. A simple rule would therefore still leave a reviewer with far more pages than can realistically be inspected.

Review priority may also depend on several interacting signals, including search demand, impression volume, position, CTR, freshness, content age, engagement and content context. A page with moderate decline but high demand may deserve greater priority than a page with severe decline but almost no exposure. These interactions may be difficult to express well with a small set of fixed thresholds.

Therefore, I will keep a transparent fixed-rule method as the baseline and test whether a data-driven scoring method produces a more useful top-K ranking. ML only earns its place if it improves the ranking enough to justify the additional complexity.

In [8]:
rule_check = pd.DataFrame(index=lane_df.index)

rule_check["declining_with_demand"] = (
    lane_df["is_declining_proxy"].eq(1)
    & lane_df["impressions_90d"].ge(100)
)

rule_check["stale_visible"] = (
    lane_df["days_since_last_update"].ge(180)
    & lane_df["impressions_90d"].ge(500)
)

rule_check["visible_low_ctr"] = (
    lane_df["impressions_90d"].ge(500)
    & lane_df["avg_position"].between(1, 20)
    & lane_df["ctr"].lt(0.5)
)

rule_summary = pd.DataFrame({
    "count": rule_check.sum(),
    "percent_of_pages": rule_check.mean() * 100
})

rule_summary.round(2)


,count,percent_of_pages
declining_with_demand,13152,43.84
stale_visible,17,0.06
visible_low_ctr,9745,32.48


In [9]:
rule_check["number_of_rules_triggered"] = rule_check.sum(axis=1)

rule_check["number_of_rules_triggered"].value_counts().sort_index()

,count
number_of_rules_triggered,
0,13208
1,10680
2,6102
3,10


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.